In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings ,  ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate

C:\Users\karan\AppData\Local\Temp\ipykernel_4572\1717129962.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [4]:
loader = PyPDFLoader("../data/data.pdf")
docs = loader.load()
len(docs)


9

In [6]:
splitter =  RecursiveCharacterTextSplitter(chunk_size=1000 , chunk_overlap=200)
splitted_data   =  splitter.split_documents(docs)
len(splitted_data)

18

In [7]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview" ,  output_dimensionality=768,)

In [8]:
vector_store=  Chroma.from_documents(
    documents=splitted_data,
    embedding=embeddings
)

In [11]:
query ="Machine Learning "
data = vector_store.similarity_search(query=query , k=2)
print(data)

[Document(id='81fccb78-72aa-45f6-9e04-8b11eacf2e8c', metadata={'title': 'ELC-112 — Principles of Digital Electronics', 'source': '../data/data.pdf', 'page': 0, 'total_pages': 9, 'producer': 'Skia/PDF m154 Google Docs Renderer', 'creationdate': '', 'creator': 'PyPDF', 'page_label': '1'}, page_content='10\n \nEncoder\n \n/\n \nDecoder\n \n🟠\n \n8/10\n \nMedium-High\n \n11\n \nIC\n \n7447\n \n/\n \nIC\n \n74138\n \n🟡\n \n7/10\n \nMedium\n \n—\n \nshort-note\n \ntype\n \n12\n \nParity\n \nBit\n \n/\n \nError\n \nDetection\n \n🟡\n \n7/10\n \nMedium\n \n—\n \nshort-note\n \ntype'), Document(id='12e13ad7-af2f-44d4-beb7-6972fa94e0be', metadata={'producer': 'Skia/PDF m154 Google Docs Renderer', 'source': '../data/data.pdf', 'total_pages': 9, 'creationdate': '', 'title': 'ELC-112 — Principles of Digital Electronics', 'page': 7, 'creator': 'PyPDF', 'page_label': '8'}, page_content="🔴\n \nTier\n \n1\n \n—\n \nDon't\n \nSkip\n \n●\n \nState\n \nand\n \nprove\n \nDe\n \nMorgan's\n \ntheorems.\n \n●\

In [12]:
context = ""
for doc in data:
    context += doc.page_content + "\n"

In [14]:
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')

In [15]:
def get_context(query:str):
    data = vector_store.similarity_search(query=query)
    context = ""
    for doc in data:
        context += doc.page_content + "\n"

    return {
        "context":context,
        "question":query
    }

In [16]:
prompt= PromptTemplate.from_template(
    '''
      You are a helpful assistant and provide answerd based on the context for user question. and 
    if you don't know the answer, then you can say that 'I dont know.'
    Context: {context}
    Question: {question}
'''
)

In [17]:
chain = get_context |  prompt | llm

In [25]:
res=  chain.invoke("What are high Probability question")

In [26]:
print( res.content)

Based on the context provided, here are the high probability questions, categorized by their priority:

**🔥 Very High Priority (Universal & Derived Gates - 8.5/10):**

**Exact questions:**
*   "Write a short note on universal gates."
*   "What is meant by derived gate? Draw the symbol of any two derived gates with their Boolean expression."

**Prepare these questions:**
*   Define universal gate.
*   Why NAND is universal.
*   Why NOR is universal.
*   Implement NOT using NAND.
*   Implement AND using NAND.
*   Implement OR using NAND.
*   Implement NOT using NOR.
*   Implement AND using NOR.
*   Implement OR using NOR.

**🟠 Important Priority (Boolean Algebra + Logic Gates - 10/10):**

**Exact questions:**
*   "Using rules of Boolean algebra simplify M = XYZ + XYZ + XYZ + XYZ."
*   "State and prove De-morgan’s Theorem."
*   "Draw the logic symbol and truth table of AND & OR gate."
*   "What is meant by derived gate? Draw the symbol of any two derived gates with their Boolean expressio